In [ ]:


%load_ext autoreload
%autoreload 2

In [11]:
import numpy as np

In [12]:
from baum_welch import *
from data_processing import *

In [13]:
import pandas as pd
import ast

### read in the data

In [ ]:
data = pd.read_csv("data/sample_data.csv")

In [ ]:
data["possession_id"] = data.groupby(["gameId","playId"]).ngroup()
data.sort_values(["nflId","nflId_pr","time"]).to_csv("data/sample_data.csv", index = False)

### fitted params

In [14]:
fitted_params = pd.read_csv("fitted_params.csv")
fitted_params["tau"] = fitted_params["tau"].apply(lambda x: np.array(ast.literal_eval(x.replace("\n",","))))

In [16]:
position_df_list = []
for position in ["C", "RB", "TE", "WR", "FB", "G", "T"]:
    weeks_df = []
    for l in range(8):
        data = pd.read_csv(f"data/sample_data_week_{l}.csv")
        data = data[data["officialPosition_pb"] == position]
        fitted_dict = {}
        pass_blocker_index_ids = {}
        pass_rusher_index_ids = {}
        fitted_param_position = fitted_params[fitted_params["position"] == position]
        tau_hat = fitted_param_position["tau"].iloc[0]
        sigma_hat = fitted_param_position["sigma"].iloc[0]
        rho_hat = fitted_param_position["rho"].iloc[0]
        for key, grouped_data in data.groupby("possession_id"):
            frame_start = grouped_data.frameId.min()
            try:
                B, O, D = possession_to_voxel(grouped_data)
                j = D.shape[1]
                k = O.shape[1]
                starting_state_distribution = np.ones((j, k)) / k
                I,_ = expectation_matchup_possession(tau_hat, sigma_hat, rho_hat, starting_state_distribution, D, O, B )
                fitted_dict[key] = {"data":I, "frame":frame_start}
            except Exception as e:
                print(f"Error: {e} on example {key}")

        for key, grouped_data in data.groupby("possession_id"):
            pass_blocker_index_ids[key] = {}
            pass_rusher_index_ids[key] = {}
            for x, player_df in enumerate(grouped_data.groupby("nflId")):
                pass_blocker_index_ids[key][x] = player_df[0]
            for z, player_df in enumerate(grouped_data.groupby("nflId_pr")):
                pass_rusher_index_ids[key][z] = player_df[0]
        all_pos_df_list = []
        for possession in fitted_dict:
            pos_df_list = []
            assignments = fitted_dict[possession]["data"]
            frame_start = fitted_dict[possession]["frame"]
            pass_rusher_indexes = pass_rusher_index_ids[possession]
            pass_blocker_indexes = pass_blocker_index_ids[possession]
            for i in range(assignments.shape[0]):
                cur_assignment = assignments[i,:,:]
                pos_df_raw = pd.DataFrame(cur_assignment, columns = pass_rusher_indexes.values(), index=pass_blocker_indexes.values()).rename_axis("nflId").reset_index().melt(id_vars=["nflId"], value_name = "prob_assigned", var_name = "nflId_pr")
                pos_df_raw["frame_id"] = i + frame_start
                pos_df_raw["possession_id"] = possession
                pos_df_list.append(pos_df_raw)
            all_pos_df_list.append(pd.concat(pos_df_list))
        final_assignment_df = pd.concat(all_pos_df_list)
        final_assignment_df["week"] = l + 1
        weeks_df.append(final_assignment_df)
    position_df_list.append(pd.concat(weeks_df))
    

KeyboardInterrupt: 

In [ ]:
pd.concat(position_df_list).to_csv("position_assignment_data.csv")

In [ ]:
assignment_data = pd.read_csv("position_assignment_data.csv")

In [ ]:
assignments[0,:,:]

In [ ]:
pass_rusher_indexes

In [ ]:
pass_blocker_indexes

In [ ]:
final_assignment_df

In [ ]:
player_data = pd.read_csv("data/players.csv")

In [ ]:
pb_merge = player_data.merge(final_assignment_df, left_on = ["nflId"], right_on = "nflId")[["nflId","officialPosition","displayName","nflId_pr", "prob_assigned","frame_id","possession_id"]]

In [ ]:
pb_merge.rename(mapper = {"displayName":"pass_blocker_name", "officialPosition": "blocker_position"},inplace=True, axis= 1)

In [ ]:
pb_merge_2 = pb_merge.merge(player_data, how = "inner", left_on = "nflId_pr", right_on = 'nflId')[["nflId_x","officialPosition","displayName","nflId_pr", "prob_assigned","frame_id","possession_id", "blocker_position","pass_blocker_name"]]

In [ ]:
pb_merge_2.rename(mapper = {"displayName":"pass_rusher_name", "officialPosition": "rusher_position", "nflId_x":"nflId"},inplace=True, axis= 1)

In [ ]:
attention_drawn_week_1 = pb_merge_2.groupby("nflId_pr").agg({"prob_assigned": np.mean, "pass_rusher_name": pd.unique}).reset_index().sort_values("prob_assigned", ascending = False).merge(player_data, right_on = "nflId", left_on = "nflId_pr")[["nflId_pr", "pass_rusher_name", "officialPosition", "prob_assigned"]]

In [ ]:
attention_drawn_week_1.groupby("officialPosition").apply(lambda x: x.sort_values("prob_assigned", ascending = False).head(5))

In [ ]:
attention_drawn_frame = assignment_data.groupby(["nflId_pr","possession_id", "frame_id", "week"]).agg(prob_assigned=("prob_assigned",sum), num_blockers =("nflId", len)).reset_index()

In [ ]:
attention_drawn_frame["prop_assigned"] = attention_drawn_frame["prob_assigned"] / attention_drawn_frame["num_blockers"]

In [ ]:
max_attention_drawn_possession = attention_drawn_frame.groupby(["nflId_pr","possession_id", "week"]).agg({"prop_assigned": "max"}).reset_index()

In [ ]:
attention_df = max_attention_drawn_possession.groupby(["nflId_pr"]).agg(num_snaps = ("prop_assigned", len), avg_peak_attention = ("prop_assigned", np.mean)).reset_index().merge(player_data, left_on = "nflId_pr", right_on = "nflId")[["nflId_pr", "officialPosition", "displayName", "num_snaps", "avg_peak_attention"]]

In [ ]:
attention_df.rename(mapper = {"displayName":"rusher_name"}, inplace = True, axis = 1)

In [ ]:
attention_df.head()

In [ ]:
attention_drawn_frame.head()

In [ ]:
max_attention_drawn_possession

In [ ]:
attention_df.sort_values(["officialPosition", "num_snaps", "avg_peak_attention"], ascending = [False, False, False])

In [ ]:
attention_df.groupby("officialPosition").apply(lambda x: x[x.num_snaps >= 200].sort_values("avg_peak_attention", ascending = False).head(5))

In [ ]:
attention_df.groupby("nflId_pr")["avg_peak_attention"].transform(lambda x: pd.rank(x, method = "dense"))

In [ ]:
sample_data = pd.read_csv("data/sample_data.csv")

In [ ]:
sample_data_columns = sample_data.columns.values

In [ ]:
position_sample = pd.merge(player_data, sample_data, left_on = ["nflId"], right_on = ["nflId"], how = "inner")[sample_data_columns.tolist() + ["officialPosition"]]

In [ ]:
position_sample.to_csv("data/position_sample_data.csv", index = False)

In [ ]:
scouting_data = pd.read_csv("data/pffScoutingData.csv")

In [ ]:
position_sample.merge(scouting_data, left_on = ["gameId", "playId", "nflId"], right_on = ["gameId", "playId", "nflId"] )[position_sample.columns.values.tolist() + ["pff_role", "pff_positionLinedUp"]].to_csv("data/position_sample_data.csv", index = False)

In [ ]:
position_sample["officialPosition"].unique()

In [ ]:
position_sample = pd.read_csv("data/position_sample_data.csv")

In [ ]:
position_sample[(position_sample["pff_role"] == "Pass Block") & (position_sample["officialPosition"] == "T")]

In [ ]:
[np.fromstring(item) for item in .split("\n")]

In [ ]:
np.fromstring(fitted_params["tau"][0], sep = "\n")

In [ ]:
np.array(ast.literal_eval(fitted_params["tau"][0].replace("\n",",")))

In [ ]:
fitted_params["tau"][0]

In [ ]:
fitted_params

### Filtering

In [17]:
from scipy.stats import norm
position_df_list = []
for position in ["C", "RB", "TE", "WR", "FB", "G", "T"]:
    weeks_df = []
    for l in range(8):
        data = pd.read_csv(f"data/sample_data_week_{l}.csv")
        data = data[data["officialPosition_pb"] == position]
        fitted_dict = {}
        pass_blocker_index_ids = {}
        pass_rusher_index_ids = {}
        fitted_param_position = fitted_params[fitted_params["position"] == position]
        tau_hat = fitted_param_position["tau"].iloc[0]
        sigma_hat = fitted_param_position["sigma"].iloc[0]
        rho_hat = fitted_param_position["rho"].iloc[0]
        for key, grouped_data in data.groupby("possession_id"):
            try:
                forward_list = []
                B, O, D = possession_to_voxel(grouped_data)

                j = D.shape[1]
                t, k, _ = O.shape
                B = np.repeat(
                        B[:, np.newaxis, :], k, axis=-2
                    )
                O_B_stacked = np.stack(
                        (O, B), axis=-1
                    )  ## new matrix with 4 dimensions (t x 2 x k x 2)
                expected_centroid = np.squeeze(
                        np.matmul(O_B_stacked, tau_hat)
                    ) 
                transition_matrix = np.zeros((k, k))  ### state transition matrix
                np.fill_diagonal(transition_matrix, rho_hat)
                transition_matrix[transition_matrix == 0] = (1 - rho_hat) / (k - 1)
                starting_state_distribution = np.ones((j, k)) / k
                for lineman in range(j):
                    new_D = D[:,lineman,:]
                    new_D = np.repeat(
                        new_D[:, np.newaxis, :], k, axis=-2
                    )  ### replicate individual lineman k times
                      ## extend ballcarrier k times B has same shape as O

                    
                     ## gets convex combination of expected offensive lineman centroid for each possible defender
                    pdf_location_difference = norm(loc=expected_centroid, scale=np.sqrt(sigma_hat)).pdf(
                        new_D
                    )

                    pdf_location_difference = np.prod(
                        pdf_location_difference, -1
                    )  ## since x,y independent normal we multily their densities

                    
                    forward_result = forward_procedure(
                        pdf_location_difference, starting_state_distribution[lineman,:][np.newaxis,:], transition_matrix, t
                    )
                    forward_list.append(forward_result)
                fitted_dict[key] = {"data":np.stack(forward_list, axis = 1), "frame": grouped_data.frameId.min()}
            except Exception as e:
                print(f"error {e} on possession {key}" )



        for key, grouped_data in data.groupby("possession_id"):
            pass_blocker_index_ids[key] = {}
            pass_rusher_index_ids[key] = {}
            for x, player_df in enumerate(grouped_data.groupby("nflId")):
                pass_blocker_index_ids[key][x] = player_df[0]
            for z, player_df in enumerate(grouped_data.groupby("nflId_pr")):
                pass_rusher_index_ids[key][z] = player_df[0]
        all_pos_df_list = []
        for possession in fitted_dict:
            pos_df_list = []
            assignments = fitted_dict[possession]["data"]
            frame_start = fitted_dict[possession]["frame"]
            pass_rusher_indexes = pass_rusher_index_ids[possession]
            pass_blocker_indexes = pass_blocker_index_ids[possession]
            for i in range(assignments.shape[0]):
                cur_assignment = assignments[i,:,:]
                pos_df_raw = pd.DataFrame(cur_assignment, columns = pass_rusher_indexes.values(), index=pass_blocker_indexes.values()).rename_axis("nflId").reset_index().melt(id_vars=["nflId"], value_name = "prob_assigned", var_name = "nflId_pr")
                pos_df_raw["frame_id"] = i + frame_start
                pos_df_raw["possession_id"] = possession
                pos_df_list.append(pos_df_raw)
            all_pos_df_list.append(pd.concat(pos_df_list))
        final_assignment_df = pd.concat(all_pos_df_list)
        final_assignment_df["week"] = l + 1
        weeks_df.append(final_assignment_df)
    position_df_list.append(pd.concat(weeks_df))
    

/var/folders/d4/h18vf05j4jsgmvd2j5k8nl840000gn/T/ipykernel_48856/749400653.py:33: RuntimeWarning: divide by zero encountered in scalar divide
  transition_matrix[transition_matrix == 0] = (1 - rho_hat) / (k - 1)


error could not broadcast input array from shape (52,) into shape (1,) on possession 667


/var/folders/d4/h18vf05j4jsgmvd2j5k8nl840000gn/T/ipykernel_48856/749400653.py:33: RuntimeWarning: divide by zero encountered in scalar divide
  transition_matrix[transition_matrix == 0] = (1 - rho_hat) / (k - 1)


error could not broadcast input array from shape (25,) into shape (1,) on possession 272


/var/folders/d4/h18vf05j4jsgmvd2j5k8nl840000gn/T/ipykernel_48856/749400653.py:33: RuntimeWarning: divide by zero encountered in scalar divide
  transition_matrix[transition_matrix == 0] = (1 - rho_hat) / (k - 1)


...

error could not broadcast input array from shape (25,) into shape (1,) on possession 272


/var/folders/d4/h18vf05j4jsgmvd2j5k8nl840000gn/T/ipykernel_48856/749400653.py:33: RuntimeWarning: divide by zero encountered in scalar divide
  transition_matrix[transition_matrix == 0] = (1 - rho_hat) / (k - 1)


error could not broadcast input array from shape (52,) into shape (1,) on possession 667


/var/folders/d4/h18vf05j4jsgmvd2j5k8nl840000gn/T/ipykernel_48856/749400653.py:33: RuntimeWarning: divide by zero encountered in scalar divide
  transition_matrix[transition_matrix == 0] = (1 - rho_hat) / (k - 1)


error could not broadcast input array from shape (25,) into shape (1,) on possession 272


In [18]:
assignment_filter_data = pd.concat(position_df_list)

In [19]:
assignment_filter_data.to_csv("assigment_data_filter.csv",index=False)

In [21]:
assigned_by_blocker = assignment_filter_data.groupby(["nflId_pr","possession_id", "week", "nflId"]).agg(prob_assigned=("prob_assigned",sum), ).reset_index()

In [22]:
assigned_total = assigned_by_blocker.groupby(["nflId_pr","possession_id","week"]).agg(prob_assigned_total = ("prob_assigned", sum)).reset_index()

In [25]:
assigned_df = assigned_total.merge(assigned_by_blocker)

In [26]:
assigned_df["share_assigned"] = assigned_df["prob_assigned"]/assigned_df["prob_assigned_total"]

In [27]:
assigned_df

,nflId_pr,possession_id,week,prob_assigned_total,nflId,prob_assigned,share_assigned
0,33131,60,7,157.088928,38553,45.049304,0.286776
1,33131,60,7,157.088928,41619,7.071625,0.045017
2,33131,60,7,157.088928,42500,26.288919,0.167351
3,33131,60,7,157.088928,43045,18.887202,0.120233
4,33131,60,7,157.088928,47794,5.702678,0.036302
...,...,...,...,...,...,...,...
196328,53999,927,4,38.189118,41321,0.551781,0.014449
196329,53999,927,4,38.189118,43444,0.160523,0.004203
196330,53999,927,4,38.189118,52543,26.987943,0.706692
196331,53999,927,4,38.189118,53516,5.333175,0.139652


In [30]:
timesteps_df = assignment_filter_data.groupby(["nflId","possession_id","week","nflId_pr"]).apply(lambda x: len(x)).reset_index()

In [31]:
timesteps_df.rename(mapper= {0:"timesteps"}, axis = 1, inplace = True)

In [34]:
assigned_df_time = timesteps_df.merge(assigned_df)

In [35]:
assigned_df_time["time_to_event"] = assigned_df_time["timesteps"] / 10 

In [87]:
assigned_df_time.merge(pff_map[["gameId","possession_id","week","playId"]]).merge(plays)

KeyboardInterrupt: 

In [44]:
plays["passResult"].unique()

array(['I', 'C', 'S', 'R', 'IN'], dtype=object)

In [ ]:
attention_drawn_frame["prop_assigned"] = attention_drawn_frame["prob_assigned"] / attention_drawn_frame["num_blockers"]

In [ ]:
max_attention_drawn_possession = attention_drawn_frame.groupby(["nflId_pr","possession_id", "week"]).agg({"prop_assigned": "max"}).reset_index()

In [ ]:
attention_df = max_attention_drawn_possession.groupby(["nflId_pr"]).agg(num_snaps = ("prop_assigned", len), avg_peak_attention = ("prop_assigned", np.mean)).reset_index().merge(player_data, left_on = "nflId_pr", right_on = "nflId")[["nflId_pr", "officialPosition", "displayName", "num_snaps", "avg_peak_attention"]]

In [ ]:
attention_df.rename(mapper = {"displayName":"rusher_name"}, inplace = True, axis = 1)

In [ ]:
attention_df.groupby("officialPosition").apply(lambda x: x[x.num_snaps >= 150].sort_values("avg_peak_attention", ascending = False).head(5))

### Creating survival data

In [1]:
import pandas as pd

In [2]:
data = pd.read_csv("assigment_data_filter.csv")

In [3]:
plays = pd.read_csv("data/plays.csv")

In [4]:
pff_map = pd.read_csv("pff_mapped_poss_id.csv")

In [5]:
pff_map["week"] += 1

In [6]:
week_df  = pd.concat([pd.read_csv(f"data/week{i+1}.csv") for i in range(8)])

In [7]:
ball_snap_time = week_df[week_df.event == "ball_snap"].groupby(["gameId","playId"]).apply(lambda x: x.time.unique()[0]).reset_index()

In [8]:
ball_snap_time.rename(mapper = {0:"snap_time"}, axis = 1, inplace = True)

In [9]:
week_df.event.uni

array(['None', 'ball_snap', 'autoevent_passforward', 'pass_forward',
       'autoevent_ballsnap', 'line_set', 'play_action', 'pass_arrived',
       'autoevent_passinterrupted', 'fumble', 'fumble_offense_recovered',
       'qb_sack', 'run', 'man_in_motion', 'pass_outcome_caught',
       'pass_outcome_incomplete', 'pass_tipped', 'qb_strip_sack', 'shift',
       'first_contact', 'huddle_break_offense', 'lateral', 'handoff',
       'penalty_flag', 'tackle', 'dropped_pass', 'out_of_bounds'],
      dtype=object)

In [ ]:
del week_df

In [366]:
adjusted_week_df =  pd.concat([pd.read_csv(f"data/sample_data_week_{i}.csv") for i in range(8)])

In [ ]:
snaps_df = pd.merge(ball_snap_time, adjusted_week_df)

In [ ]:
snaps_filtered_df = snaps_df[snaps_df.snap_time <= snaps_df.time]

In [ ]:
del snaps_df
del adjusted_week_df

In [ ]:
mapped_poss_data = pff_map[["possession_id","week","gameId","playId"]].merge(data, how = "inner", left_on = ["possession_id", "week"], right_on = ["possession_id","week"])

In [ ]:
del data

In [ ]:
snaps_filtered_df[["gameId","playId","snap_time","time", "frameId"]].merge(mapped_poss_data, left_on = ["frameId","playId","gameId"], right_on=["frame_id","playId","gameId"], how = "inner")

In [55]:
snaps = week_df.groupby(["gameId","playId","event"]).apply(lambda x: x.time.unique()).reset_index()

In [51]:
pass_forward = week_df.groupby(["gameId","playId"]).apply(lambda x: x[x.event == "pass_forward"].time.unique()).reset_index()

In [58]:
snaps_of_interest = snaps[snaps.event.isin(["ball_snap","pass_forward","qb_strip_sack","run","qb_sack"])]

In [59]:
snaps_of_interest["new_event"] = snaps_of_interest["event"].apply(lambda x: "decision" if x in ["run","pass_forward"] else x)

/var/folders/d4/h18vf05j4jsgmvd2j5k8nl840000gn/T/ipykernel_48856/42445599.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  snaps_of_interest["new_event"] = snaps_of_interest["event"].apply(lambda x: "decision" if x in ["run","pass_forward"] else x)


In [61]:
snaps_of_interest["new_event"] = snaps_of_interest["new_event"].apply(lambda x: "censor" if x in ["qb_sack","qb_strip_sack"] else x)

/var/folders/d4/h18vf05j4jsgmvd2j5k8nl840000gn/T/ipykernel_48856/950610883.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  snaps_of_interest["new_event"] = snaps_of_interest["new_event"].apply(lambda x: "censor" if x in ["qb_sack","qb_strip_sack"] else x)


In [63]:
snaps_of_interest.rename(axis = 1, inplace = True, mapper = {0:"timestamp"})

/var/folders/d4/h18vf05j4jsgmvd2j5k8nl840000gn/T/ipykernel_48856/2957275738.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  snaps_of_interest.rename(axis = 1, inplace = True, mapper = {0:"timestamp"})


In [66]:
snaps_of_interest["timestamp"] = snaps_of_interest["timestamp"].apply(lambda x: x[0])

/var/folders/d4/h18vf05j4jsgmvd2j5k8nl840000gn/T/ipykernel_48856/978658405.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  snaps_of_interest["timestamp"] = snaps_of_interest["timestamp"].apply(lambda x: x[0])


In [70]:
snaps_of_interest["timestamp"] = pd.to_datetime(snaps_of_interest["timestamp"])

/var/folders/d4/h18vf05j4jsgmvd2j5k8nl840000gn/T/ipykernel_48856/2327402048.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  snaps_of_interest["timestamp"] = pd.to_datetime(snaps_of_interest["timestamp"])


In [322]:
snaps_of_interest.groupby(["gameId","playId"]).apply(lambda x: len(x)).reset_index().sum()

8531.0

In [323]:
len(snaps_of_interest)

17062

In [77]:
snaps_time_event = snaps_of_interest.groupby(["gameId","playId"]).apply(lambda x: x.sort_values("timestamp").timestamp.diff()).reset_index()[["gameId","playId","timestamp"]]

In [78]:
snaps_time_event["time_length"] = snaps_time_event["timestamp"].apply(lambda x: x.total_seconds())

In [85]:
survival_data = pd.merge(snaps_time_event[~snaps_time_event.time_length.isna()][["time_length","gameId","playId"]], snaps_of_interest[snaps_of_interest.new_event != "ball_snap"], how = "inner", left_on = ["gameId","playId"],right_on = ["gameId","playId"])

In [86]:
survival_data

,time_length,gameId,playId,event,timestamp,new_event
0,3.4,2021090900,97,pass_forward,2021-09-10 00:26:35.000,decision
1,2.5,2021090900,137,pass_forward,2021-09-10 00:28:13.000,decision
2,2.2,2021090900,187,pass_forward,2021-09-10 00:29:17.700,decision
3,3.2,2021090900,282,pass_forward,2021-09-10 00:31:55.300,decision
4,2.6,2021090900,349,pass_forward,2021-09-10 00:34:08.200,decision
...,...,...,...,...,...,...
8514,4.8,2021110100,4310,qb_sack,2021-11-02 03:15:57.700,censor
8515,3.7,2021110100,4363,pass_forward,2021-11-02 03:18:44.700,decision
8516,4.1,2021110100,4392,qb_sack,2021-11-02 03:19:22.000,censor
8517,2.6,2021110100,4411,pass_forward,2021-11-02 03:19:42.800,decision


In [83]:
snaps_time_event

,gameId,playId,timestamp,time_length
0,2021090900,97,NaT,NaN
1,2021090900,97,0 days 00:00:03.400000,3.4
2,2021090900,137,NaT,NaN
3,2021090900,137,0 days 00:00:02.500000,2.5
4,2021090900,187,NaT,NaN
...,...,...,...,...
17057,2021110100,4392,0 days 00:00:04.100000,4.1
17058,2021110100,4411,NaT,NaN
17059,2021110100,4411,0 days 00:00:02.600000,2.6
17060,2021110100,4433,NaT,NaN


In [90]:
assigned_df_time[["nflId","possession_id","week","nflId_pr","share_assigned"]].merge(pff_map[["possession_id","week","playId","gameId"]]).merge(survival_data)

,nflId,possession_id,week,nflId_pr,share_assigned,playId,gameId,time_length,event,timestamp,new_event
0,29550,145,2,43350,0.000013,59,2021091901,5.9,pass_forward,2021-09-19 17:03:25.800,decision
1,29550,145,2,43350,0.000013,59,2021091901,5.9,pass_forward,2021-09-19 17:03:25.800,decision
2,29550,145,2,43350,0.000013,59,2021091901,5.9,pass_forward,2021-09-19 17:03:25.800,decision
3,29550,145,2,43350,0.000013,59,2021091901,5.9,pass_forward,2021-09-19 17:03:25.800,decision
4,29550,145,2,43350,0.000013,59,2021091901,5.9,pass_forward,2021-09-19 17:03:25.800,decision
...,...,...,...,...,...,...,...,...,...,...,...
3771037,53861,469,2,53459,0.018573,4175,2021091905,2.7,pass_forward,2021-09-19 20:17:23.900,decision
3771038,53861,469,2,53459,0.018573,4175,2021091905,2.7,pass_forward,2021-09-19 20:17:23.900,decision
3771039,53861,469,2,53459,0.018573,4175,2021091905,2.7,pass_forward,2021-09-19 20:17:23.900,decision
3771040,53861,469,2,53459,0.018573,4175,2021091905,2.7,pass_forward,2021-09-19 20:17:23.900,decision


In [ ]:
ssigned_df_time[["nflId","possession_id","week","nflId_pr","share_assigned"]].merge(pff_map[["possession_id","week","playId","gameId"]])

In [93]:
pff_new_map = pff_map[["gameId","playId","possession_id","week"]].drop_duplicates()

In [100]:
assigned_df_time[["nflId","possession_id","week","nflId_pr","share_assigned"]].merge(pff_new_map[["possession_id","week","playId","gameId"]]).merge(survival_data).merge(plays[["gameId","playId","offenseFormation","down", "yardsToGo", "pff_passCoverage","pff_"]])

,nflId,possession_id,week,nflId_pr,share_assigned,playId,gameId,time_length,event,timestamp,new_event,offenseFormation,down,yardsToGo,pff_passCoverage,pff_playAction
0,29550,145,2,43350,0.000013,59,2021091901,5.9,pass_forward,2021-09-19 17:03:25.800,decision,SINGLEBACK,1,10,Cover-1,1
1,29550,145,2,43455,0.004685,59,2021091901,5.9,pass_forward,2021-09-19 17:03:25.800,decision,SINGLEBACK,1,10,Cover-1,1
2,29550,145,2,44877,0.000625,59,2021091901,5.9,pass_forward,2021-09-19 17:03:25.800,decision,SINGLEBACK,1,10,Cover-1,1
3,29550,145,2,45226,0.245782,59,2021091901,5.9,pass_forward,2021-09-19 17:03:25.800,decision,SINGLEBACK,1,10,Cover-1,1
4,29550,145,2,46146,0.570519,59,2021091901,5.9,pass_forward,2021-09-19 17:03:25.800,decision,SINGLEBACK,1,10,Cover-1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
171406,53471,469,2,53459,0.850286,4175,2021091905,2.7,pass_forward,2021-09-19 20:17:23.900,decision,SHOTGUN,3,3,Cover-3,0
171407,53861,469,2,43319,0.181533,4175,2021091905,2.7,pass_forward,2021-09-19 20:17:23.900,decision,SHOTGUN,3,3,Cover-3,0
171408,53861,469,2,43638,0.501368,4175,2021091905,2.7,pass_forward,2021-09-19 20:17:23.900,decision,SHOTGUN,3,3,Cover-3,0
171409,53861,469,2,52462,0.034782,4175,2021091905,2.7,pass_forward,2021-09-19 20:17:23.900,decision,SHOTGUN,3,3,Cover-3,0


In [108]:
qb_map = plays.merge(pff_map[pff_map["pff_role"] == "Pass"][["nflId","gameId","playId"]])[["nflId","playId","gameId"]].drop_duplicates()

In [109]:
qb_map.rename(axis = 1, inplace = True, mapper = {"nflId": "qb_id"})

In [112]:
design_data = assigned_df_time[["nflId","possession_id","week","nflId_pr","share_assigned"]].merge(pff_new_map[["possession_id","week","playId","gameId"]]).merge(survival_data).merge(plays[["gameId","playId","offenseFormation","down", "yardsToGo", "pff_passCoverage","pff_playAction"]]).merge(qb_map)

In [125]:
rushers = design_data.groupby(["playId","gameId"]).apply(lambda x: list(set(x.nflId_pr))).reset_index()
rushers.rename(axis = 1, mapper = {0:"rusher_id"}, inplace = True)


In [126]:
blockers = design_data.groupby(["playId","gameId"]).apply(lambda x: list(set(x.nflId))).reset_index()
blockers.rename(axis = 1, mapper = {0:"blocker_id"}, inplace = True)


In [143]:
survival_data = blockers.merge(rushers).merge(design_data.drop_duplicates(subset=["gameId","playId"]))

In [137]:
blockers.merge(rushers)

,playId,gameId,blocker_id,rusher_id
0,54,2021091910,"[47814, 52486, 41232, 37234, 47797, 53497]","[35441, 40074, 41915, 46081]"
1,54,2021092300,"[46170, 44876, 45594, 44820, 42424, 42362, 479...","[46255, 52498, 41300, 53624, 43356]"
2,54,2021092609,"[44832, 52491, 52526, 47824, 38642, 43384, 47803]","[46249, 47786, 46204, 53053]"
3,54,2021092612,"[45056, 33107, 46237, 52477, 41310]","[44867, 52525, 43694, 42431]"
4,54,2021101011,"[52480, 37130, 38779, 39965, 42367]","[42360, 47785, 45011, 52422]"
...,...,...,...,...
7438,5087,2021101702,"[43971, 47976, 53452, 52466, 46131, 44853, 478...","[43791, 52472, 44825, 41341, 52415]"
7439,5108,2021100307,"[48513, 52553, 37266, 53466, 47805]","[46144, 52585, 43338, 43326]"
7440,5133,2021101702,"[47976, 53452, 46226, 46131, 52466, 44853, 47801]","[44825, 47799, 52415, 43791]"
7441,5153,2021100307,"[48513, 52553, 37266, 53466, 47805]","[46144, 52585, 43338, 43326]"


In [139]:
survival_data = pd.read_csv("data/survival_data.csv")

In [147]:
import itertools

In [148]:
big_list = set(itertools.chain.from_iterable(survival_data["blocker_id"].values.tolist()))

In [150]:
len(big_list)

515

In [151]:
blocker_map = {val:index for index,val in enumerate(big_list)}

In [152]:
blocker_map

{45056: 0,
 45062: 1,
 45069: 2,
 43045: 3,
 45094: 4,
 45142: 5,
 41069: 6,
 30842: 7,
 30869: 8,
 45217: 9,
 53433: 10,
 53436: 11,
 53442: 12,
 53443: 13,
 39109: 14,
 53446: 15,
 53452: 16,
 53453: 17,
 45267: 18,
 45268: 19,
 53464: 20,
 37082: 21,
 53466: 22,
 53471: 23,
 45281: 24,
 37090: 25,
 53475: 26,
 53480: 27,
 39146: 28,
 53482: 29,
 53484: 30,
 37101: 31,
 53491: 32,
 53492: 33,
 53497: 34,
 53499: 35,
 37118: 36,
 53506: 37,
 41222: 38,
 53510: 39,
 53512: 40,
 45321: 41,
 37130: 42,
 53514: 43,
 53516: 44,
 53517: 45,
 41232: 46,
 53522: 47,
 53523: 48,
 41236: 49,
 41237: 50,
 53524: 51,
 53527: 52,
 41242: 53,
 45339: 54,
 43293: 55,
 43295: 56,
 53536: 57,
 43297: 58,
 45346: 59,
 53539: 60,
 43302: 61,
 53543: 62,
 43307: 63,
 39212: 64,
 53549: 65,
 41262: 66,
 45355: 67,
 41264: 68,
 53553: 69,
 53555: 70,
 53556: 71,
 53557: 72,
 43320: 73,
 37179: 74,
 43324: 75,
 53571: 76,
 43334: 77,
 53574: 78,
 41286: 79,
 43337: 80,
 53579: 81,
 53580: 82,
 41293: 83,
 4

In [153]:
big_list_rusher = set(itertools.chain.from_iterable(survival_data["rusher_id"].values.tolist()))

In [154]:
rusher_map = {val:index for index,val in enumerate(big_list_rusher)}

In [157]:
rusher_mat = np.zeros((survival_data.shape[0],len(rusher_map)))

In [158]:
blocker_mat = np.zeros((survival_data.shape[0],len(blocker_map)))

In [160]:
for i, element in enumerate(survival_data["rusher_id"]):
    rusher_mat[i,[rusher_map[player_id] for player_id in element]] = 1

In [163]:
for i, element in enumerate(survival_data["blocker_id"]):
    blocker_mat[i,[blocker_map[player_id] for player_id in element]] = 1

In [165]:
blocker_mat *= -1

In [169]:
pd.DataFrame(np.concatenate([rusher_mat,blocker_mat],axis = 1)).to_csv("blocker_rusher_design_matrix.csv", index = False)

In [324]:
surviv_coefs = pd.read_csv("survival_coefs.csv")


In [325]:
surviv_coefs

,Unnamed: 0,betas.1,betas.2,betas.3,betas.4,betas.5,betas.6,betas.7,betas.8,betas.9,...,betas_coverage.6,betas_coverage.7,betas_coverage.8,betas_coverage.9,betas_coverage.10,betas_coverage.11,betas_coverage.12,intercept,pa,lp__
0,chain:1,0.105146,-0.120725,0.037499,-0.394411,0.11799,-0.147367,0.048722,0.244796,-0.395107,...,-0.271653,-0.452918,-0.378644,-0.457114,-0.604596,-0.402588,-0.220066,-2.777249,-0.193566,0


In [326]:
rusher_coefs = surviv_coefs.iloc[0][1:-3][0:len(rusher_map)]

In [282]:
len(rusher_map)

674

In [283]:
len(rusher_coefs)

674

In [246]:
rusher_coefs

betas.1     -1.026356
betas.2     -0.320765
betas.3      0.347535
betas.4      0.138105
betas.5      0.228001
               ...   
betas.670    0.092715
betas.671   -0.098901
betas.672   -0.379488
betas.673    0.160315
betas.674   -1.200253
Name: 0, Length: 674, dtype: object

In [327]:
 blocker_coefs = surviv_coefs.iloc[0][1:-3][len(rusher_map):1189]

In [248]:
blocker_coefs

betas.675    -0.519519
betas.676     0.241131
betas.677     0.075179
betas.678    -0.799589
betas.679     -0.22496
                ...   
betas.1185   -0.817643
betas.1186   -0.420371
betas.1187   -0.534507
betas.1188   -0.877684
betas.1189    0.503979
Name: 0, Length: 515, dtype: object

In [328]:
blocker_coef_df = pd.DataFrame(blocker_coefs).reset_index()

In [306]:
blocker_coef_df

,index,0
0,betas.675,0.416876
1,betas.676,-0.020259
2,betas.677,-1.250569
3,betas.678,0.901592
4,betas.679,-0.238356
...,...,...
510,betas.1185,-1.0652
511,betas.1186,-0.595487
512,betas.1187,-0.263412
513,betas.1188,0.485615


In [329]:
blocker_coef_df["index"] = blocker_coef_df["index"].apply(lambda x: int(x.split(".")[-1]))

In [330]:
blocker_coef_df["index"] -= 1

In [331]:
rusher_coef_df = pd.DataFrame(rusher_coefs).reset_index()
rusher_coef_df["index"] = rusher_coef_df["index"].apply(lambda x: int(x.split(".")[-1]) - 1)

In [332]:
rusher_coef_df["nflId"] = rusher_coef_df["index"].apply(lambda x: inverse_rusher_map[x])

In [188]:
inverse_rusher_map = {rusher_map[index]:index for index in rusher_map}

In [193]:
inverse_blocker_map = {blocker_map[index]:index for index in blocker_map}

In [333]:
blocker_coef_df["nflId"] = blocker_coef_df["index"].apply(lambda x: inverse_blocker_map[x - min(blocker_coef_df["index"])])

In [202]:
players = pd.read_csv("data/players.csv")

In [334]:
blocker_coef_df.merge(players).groupby("officialPosition").apply(lambda x: x.sort_values(0, ascending = False).head(5))

index         0  nflId height  weight   birthDate  \
officialPosition                                                          
C                119    793  1.489221  43433    6-4     306  1993-04-27   
                 335   1009  1.214695  52486    6-4     290  1997-11-17   
                 474   1148  1.007579  44870    6-6     320  1995-08-05   
                 491   1165  0.815345  42883    6-3     300  1992-07-10   
                 121    795  0.775203  41390    6-3     301  1991-07-27   
FB               285    959  0.422943  48233    6-1     242  1996-07-09   
                 128    802   0.25915  43465    6-1     238  1993-05-23   
                 294    968  0.214995  40078    6-1     240  1991-04-23   
                 210    884  0.178003  41808    6-0     240  1992-04-08   
                 149    823  0.105955  47749    6-3     255  1994-12-15   
G                2      676  1.968202  45069    6-5     260  1994-01-27   
                 0      674  1.316242  45056    6-5     320  1994-03-04   
                 297    971  1.120586  46235    6-4     315  1994-11-21   
                 97     771  1.010118  41321    6-3     315  1993-06-14   
                 256    930   0.98357  46119    6-5     298  1997-05-12   
RB               379   1053  0.686635  46461   5-10     205  1995-07-30   
                 117    791  0.417604  53662   5-10     205         NaN   
                 402   1076   0.31084  46526   5-11     224  1995-04-15   
                 243    917  0.275298  46100   5-11     215  1995-02-17   
                 242    916   0.26477  46096   5-11     220  1996-02-02   
T                467   1141   2.09369  44844    6-6     314  1994-04-22   
                 269    943  1.494932  46152    6-8     345  1996-05-02   
                 85     759  1.482477  41296    6-5     310  1991-10-17   
                 468   1142  1.148448  44846    6-6     320  1995-10-09   
                 46     720  1.128475  41232    6-5     309  1992-02-11   
TE               447   1121  0.671471  46797    6-5     239  1994-11-12   
                 5      679  0.459933  45142    6-3     256  1994-11-22   
                 30     704  0.435368  53484    6-5     258         NaN   
                 220    894  0.434947  48011    6-5     251  1995-07-01   
                 71     745  0.376396  53556    6-3     235         NaN   
WR               336   1010  0.613411  52489    6-3     215  1998-11-13   
                 484   1158  0.447816  44896    6-1     209  1996-02-27   
                 227    901  0.376829  48097    6-2     200  1996-11-09   
                 265    939  0.367279  39989    6-0     193  1992-04-10   
                 514   1188  0.338215  45052    6-2     225  1996-01-06   

                               collegeName officialPosition       displayName  
officialPosition                                                               
C                119              Missouri                C   Connor McGovern  
                 335                Temple                C     Matt Hennessy  
                 474       Louisiana State                C       Ethan Pocic  
                 491               Georgia                C     David Andrews  
                 121            Ohio State                C     Corey Linsley  
FB               285             Wisconsin               FB       Alec Ingold  
                 128              Nebraska               FB     Andy Janovich  
                 294               Harvard               FB     Kyle Juszczyk  
                 210        San Jose State               FB       Keith Smith  
                 149             Tennessee               FB     Jakob Johnson  
G                2         San Diego State                G  Daniel Brunskill  
                 0                  Baylor                G       Kyle Fuller  
                 297         Virginia Tech                G      Wyatt Teller  
                 97        Louisiana State    

In [335]:
rusher_coef_df.merge(players).groupby("officialPosition").apply(lambda x: x.sort_values(0, ascending = False).head(5))

index         0  nflId height  weight   birthDate  \
officialPosition                                                          
CB               590    590  0.620445  40688    6-0     191  1991-08-16   
                 340    340  0.567434  40008   5-10     186  1988-11-01   
                 549    549   0.55102  46652   5-11     189  1996-08-07   
                 110    110  0.447558  43366    6-1     215  1995-02-22   
                 210    210  0.415342  43712    6-0     198  1991-10-14   
DE               99      99  0.766111  43352    6-4     287  1992-09-23   
                 169    169  0.755712  41464    6-2     290  1991-08-11   
                 453    453   0.73024  48462    6-4     285  1996-08-05   
                 389    389  0.677089  46255    6-2     242  1995-12-11   
                 21      21  0.666393  45274    6-5     300  1994-04-12   
DT               309    309  0.848466  39959    6-3     294  1990-11-29   
                 620    620  0.828204  44867    6-3     318  1994-02-28   
                 293    293  0.742722  48123    6-1     285  1997-03-01   
                 248    248  0.689264  43787    6-5     294  1993-07-23   
                 62      62  0.633999  43301    6-2     305  1994-04-02   
FS               438    438  0.512214  52512    6-0     198  1997-08-18   
                 223    223  0.328488  47830    6-2     195  1997-07-18   
                 149    149  0.305842  53674    6-0     180         NaN   
                 600    600  0.298383  44827    6-1     212  1996-04-02   
                 477    477  0.283336  48508    6-0     209  1996-12-05   
G                393    393  0.351637  46267    6-3     320  1997-01-03   
ILB              164    164  0.516102  45533    6-0     231  1994-02-21   
                 145    145  0.406576  53663    6-2     251         NaN   
                 144    144  0.331171  43420    6-2     237  1994-01-09   
                 128    128   0.32533  43388    6-1     245  1993-11-06   
                 360    360  0.298908  46191    6-1     235  1994-11-15   
LB               156    156  0.182108  53681    6-4     245         NaN   
MLB              655    655   0.36194  44974    6-1     230  1995-08-08   
                 67      67  0.333747  43306    6-1     216  1995-07-26   
                 299    299   0.29352  46085    6-5     250  1998-05-02   
                 242    242  0.228939  47872    6-1     234  1996-07-29   
                 78      78  0.218654  43325    6-1     244  1995-09-03   
NT               197    197  0.701197  43694    6-0     340  1992-11-06   
                 418    418  0.693117  52464    6-7     310         NaN   
                 193    193  0.685133  47786    6-3     303  1997-12-21   
                 574    574  0.533328  40608    6-2     293  1989-02-06   
                 329    329  0.501126  39997    6-3     340  1992-03-30   
OLB              292    292  0.732436  46074    6-4     275  1996-06-24   
                 591    591  0.721187  52979    6-3     275  1997-06-12   
                 202    202  0.694683  47795    6-5     277  1997-12-03   
                 135    135  0.667596  53639    6-3     265         NaN   
                 589    589  0.619109  52971    6-3     255         NaN   
RB               515    515  0.115102  53612    5-9     195         NaN   
SS               307    307  0.497027  46097    6-1     217  1997-01-20   
                 556    556  0.409768  46680   5-10     205  1996-04-27   
                 64      64   0.24424  43303   5-10     200  1993-09-08   
                 141    141  0.238553  43407    6-0     212  1993-08-04   
                 161    161  0.238256  43456   5-11     207  1994-10-27   

                                 collegeName officialPosition  \
officialPosition                                                
CB               590         Central Florida               CB   
                 340  Southeastern Louisiana               CB   
           

In [314]:
qb_coef = surviv_coefs.iloc[0][1190:1190+55].reset_index()

In [315]:
qb_coef["index"] = qb_coef["index"].apply(lambda x: int(x.split(".")[-1]))

In [316]:
qb_coef["index"] -= 1


In [317]:
qb_coef["nflId"] = qb_coef["index"].apply(lambda x: qb_inverse_map[x] )

In [318]:
qb_coef.merge(players).sort_values(0,ascending = True)

,index,0,nflId,height,weight,birthDate,collegeName,officialPosition,displayName
21,21,-0.709108,52530,6-6,227,1997-11-17,Washington,QB,Jacob Eason
10,10,-0.618674,43291,6-5,237,1992-12-30,North Dakota State,QB,Carson Wentz
3,3,-0.575237,44814,6-2,215,1994-08-20,North Carolina,QB,Mitchell Trubisky
49,49,-0.54379,46070,6-1,215,1995-04-14,Oklahoma,QB,Baker Mayfield
45,45,-0.514346,38632,6-3,202,1988-08-19,Michigan State,QB,Kirk Cousins
46,46,-0.469846,52461,6-2,218,1998-08-07,Oklahoma,QB,Jalen Hurts
17,17,-0.468704,47784,5-10,207,1997-08-07,Oklahoma,QB,Kyler Murray
26,26,-0.453746,52409,6-4,216,1996-12-10,Louisiana State,QB,Joe Burrow
5,5,-0.439158,46101,6-2,212,1997-01-07,Louisville,QB,Lamar Jackson
36,36,-0.429572,38605,5-11,215,1988-11-29,Wisconsin,QB,Russell Wilson


In [300]:
surviv_coefs.iloc[0][1190+55:-3].reset_index()

,index,0
0,betas_coverage.1,-0.849759
1,betas_coverage.2,-0.878227
2,betas_coverage.3,-1.029295
3,betas_coverage.4,-1.047714
4,betas_coverage.5,-0.868129
5,betas_coverage.6,-0.641983
6,betas_coverage.7,-0.869886
7,betas_coverage.8,-0.863128
8,betas_coverage.9,-0.998038
9,betas_coverage.10,-1.23176


In [235]:
qbs = set(survival_data["qb_id"].unique())

In [236]:
qb_map = {nflid:i for i,nflid in enumerate(qbs)}

In [238]:
survival_data["qb_id_map"] = survival_data["qb_id"].apply(lambda x: qb_inverse_map[x])

In [240]:
pd.get_dummies(survival_data["qb_id_map"]).to_csv("qb_design.csv",index=False)

In [276]:
coverage = set(survival_data["pff_passCoverage"].unique())
coverage_map = {cover:i for i, cover in enumerate(coverage)}
inverse_coverage_map = {coverage_map[i]:i for i in coverage_map}
survival_data["coverage_id"] = survival_data["pff_passCoverage"].apply(lambda x: coverage_map[x])
survival_data["coverage_map"] = survival_data["coverage_id"].apply(lambda x: inverse_coverage_map[x])

In [278]:
pd.get_dummies(survival_data["coverage_id"]).to_csv("coverage_design.csv",index = False)

In [266]:
qb_inverse_map = {qb_map[nflid]:nflid for nflid in qb_map }

In [267]:
qb_inverse_map

{0: 46072,
 1: 37255,
 2: 38538,
 3: 44814,
 4: 34452,
 5: 46101,
 6: 44822,
 7: 38937,
 8: 43290,
 9: 29851,
 10: 43291,
 11: 34843,
 12: 52895,
 13: 43424,
 14: 46240,
 15: 28963,
 16: 25511,
 17: 47784,
 18: 41258,
 19: 47789,
 20: 41265,
 21: 52530,
 22: 39987,
 23: 53430,
 24: 53431,
 25: 53432,
 26: 52409,
 27: 44984,
 28: 33084,
 29: 52413,
 30: 52414,
 31: 45244,
 32: 53440,
 33: 53444,
 34: 41291,
 35: 47180,
 36: 38605,
 37: 42831,
 38: 47825,
 39: 52434,
 40: 40021,
 41: 37083,
 42: 43490,
 43: 45159,
 44: 42344,
 45: 38632,
 46: 52461,
 47: 33138,
 48: 43380,
 49: 46070,
 50: 37110,
 51: 53496,
 52: 46076,
 53: 30078,
 54: 46079}

### strain 

In [337]:
snaps

,gameId,playId,event,0
0,2021090900,97,None,"[2021-09-10T00:26:31.100, 2021-09-10T00:26:31...."
1,2021090900,97,autoevent_passforward,[2021-09-10T00:26:34.800]
2,2021090900,97,ball_snap,[2021-09-10T00:26:31.600]
3,2021090900,97,pass_forward,[2021-09-10T00:26:35.000]
4,2021090900,137,None,"[2021-09-10T00:28:09.900, 2021-09-10T00:28:10...."
...,...,...,...,...
36369,2021110100,4411,pass_forward,[2021-11-02T03:19:42.800]
36370,2021110100,4433,None,"[2021-11-02T03:20:21.200, 2021-11-02T03:20:21...."
36371,2021110100,4433,autoevent_ballsnap,[2021-11-02T03:20:21.700]
36372,2021110100,4433,ball_snap,[2021-11-02T03:20:21.800]


In [339]:
design_data

,nflId,possession_id,week,nflId_pr,share_assigned,playId,gameId,time_length,event,timestamp,new_event,offenseFormation,down,yardsToGo,pff_passCoverage,pff_playAction,qb_id
0,29550,145,2,43350,0.000013,59,2021091901,5.9,pass_forward,2021-09-19 17:03:25.800,decision,SINGLEBACK,1,10,Cover-1,1,37110
1,29550,145,2,43455,0.004685,59,2021091901,5.9,pass_forward,2021-09-19 17:03:25.800,decision,SINGLEBACK,1,10,Cover-1,1,37110
2,29550,145,2,44877,0.000625,59,2021091901,5.9,pass_forward,2021-09-19 17:03:25.800,decision,SINGLEBACK,1,10,Cover-1,1,37110
3,29550,145,2,45226,0.245782,59,2021091901,5.9,pass_forward,2021-09-19 17:03:25.800,decision,SINGLEBACK,1,10,Cover-1,1,37110
4,29550,145,2,46146,0.570519,59,2021091901,5.9,pass_forward,2021-09-19 17:03:25.800,decision,SINGLEBACK,1,10,Cover-1,1,37110
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
171406,53471,469,2,53459,0.850286,4175,2021091905,2.7,pass_forward,2021-09-19 20:17:23.900,decision,SHOTGUN,3,3,Cover-3,0,43380
171407,53861,469,2,43319,0.181533,4175,2021091905,2.7,pass_forward,2021-09-19 20:17:23.900,decision,SHOTGUN,3,3,Cover-3,0,43380
171408,53861,469,2,43638,0.501368,4175,2021091905,2.7,pass_forward,2021-09-19 20:17:23.900,decision,SHOTGUN,3,3,Cover-3,0,43380
171409,53861,469,2,52462,0.034782,4175,2021091905,2.7,pass_forward,2021-09-19 20:17:23.900,decision,SHOTGUN,3,3,Cover-3,0,43380


ValueError: Index contains duplicate entries, cannot reshape

In [346]:
snaps_of_interest["new_event"] = snaps_of_interest["new_event"].apply(lambda x: "decision" if x == "censor" else x)

/var/folders/d4/h18vf05j4jsgmvd2j5k8nl840000gn/T/ipykernel_48856/3437000286.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  snaps_of_interest["new_event"] = snaps_of_interest["new_event"].apply(lambda x: "decision" if x == "censor" else x)


In [358]:
snaps_of_interest_wide = pd.pivot_table(snaps_of_interest[["playId","gameId","timestamp","new_event"]], index = ["playId","gameId"], values=["timestamp"], columns = ["new_event"])

['timestamp_ball_snap', 'timestamp_decision']

In [361]:
snaps_of_interest_wide = snaps_of_interest_wide.sort_index(axis=1, level=1)

In [362]:
snaps_of_interest_wide.columns = [f'{x}_{y}' for x,y in snaps_of_interest_wide.columns]

In [364]:
snaps_of_interest_wide = snaps_of_interest_wide.reset_index()

In [369]:
new_adjusted_week_df = adjusted_week_df.merge(pff_new_map).merge(snaps_of_interest_wide)

In [373]:
new_adjusted_week_df = new_adjusted_week_df[(pd.to_datetime(new_adjusted_week_df.timestamp_ball_snap) <= pd.to_datetime(new_adjusted_week_df.time)) & ( pd.to_datetime(new_adjusted_week_df.time) <= pd.to_datetime(new_adjusted_week_df.timestamp_decision)) ]

In [374]:
del adjusted_week_df

/var/folders/d4/h18vf05j4jsgmvd2j5k8nl840000gn/T/ipykernel_48856/1004689740.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_adjusted_week_df["d_ij"] = np.sqrt(np.square(new_adjusted_week_df["x_pr"] - new_adjusted_week_df["x_qb"]) + np.square(new_adjusted_week_df["y_pr"] - new_adjusted_week_df["y_qb"]))


### calculate strain

In [377]:
new_adjusted_week_df["d_ij"] = np.sqrt(np.square(new_adjusted_week_df["x_pr"] - new_adjusted_week_df["x_qb"]) + np.square(new_adjusted_week_df["y_pr"] - new_adjusted_week_df["y_qb"]))

/var/folders/d4/h18vf05j4jsgmvd2j5k8nl840000gn/T/ipykernel_48856/1004689740.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_adjusted_week_df["d_ij"] = np.sqrt(np.square(new_adjusted_week_df["x_pr"] - new_adjusted_week_df["x_qb"]) + np.square(new_adjusted_week_df["y_pr"] - new_adjusted_week_df["y_qb"]))


In [381]:
new_adjusted_week_df.groupby(["gameId","playId","nflId_pr"]).apply(lambda x: (x.d_ij - x.d_ij.shift(-1))/(-.1*x.d_ij)).reset_index()

,gameId,playId,nflId_pr,level_3,d_ij
0,2021090900,97,41263,80578,-0.045535
1,2021090900,97,41263,80579,-0.091885
2,2021090900,97,41263,80580,-0.119628
3,2021090900,97,41263,80581,-0.261090
4,2021090900,97,41263,80582,-0.238871
...,...,...,...,...,...
1506215,2021102402,3081,52464,1643035,-0.957062
1506216,2021102402,3081,52464,1643036,-1.088149
1506217,2021102402,3081,52464,1643037,-1.310420
1506218,2021102402,3081,52464,1643038,-1.377096


In [387]:
.groupby(["playId","gameId"])["d_ij"].transform(lambda x: (x - x.shift(-1))/(-.1*x))

0         -0.091038
1         -0.083332
2         -0.126674
3         -0.147124
4         -0.141410
             ...   
1725900   -0.492525
1725901   -0.476391
1725902   -0.486803
1725903   -0.522284
1725904         NaN
Name: d_ij, Length: 276721, dtype: float64

In [388]:
new_adjusted_week_pr_df = new_adjusted_week_df.drop_duplicates(["gameId","playId","nflId_pr","time"])

In [389]:
new_adjusted_week_pr_df["strain"] = new_adjusted_week_pr_df.groupby(["playId","gameId"])["d_ij"].transform(lambda x: (x - x.shift(-1))/(-.1*x))

/var/folders/d4/h18vf05j4jsgmvd2j5k8nl840000gn/T/ipykernel_48856/108843687.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_adjusted_week_pr_df["strain"] = new_adjusted_week_pr_df.groupby(["playId","gameId"])["d_ij"].transform(lambda x: (x - x.shift(-1))/(-.1*x))


In [390]:
new_adjusted_week_pr_df.head()

,gameId,playId,frameId,time,nflId_pr,x_pr,y_pr,officialPosition,snap_time,nflId_qb,...,nflId,x,y,officialPosition_pb,possession_id,week,timestamp_ball_snap,timestamp_decision,d_ij,strain
0,2021090900,583,6,2021-09-10T00:42:52.700,41263,106.05,19.26,DE,2021-09-10T00:42:52.700,25511,...,35481,104.07,19.96,TE,11,1,2021-09-10 00:42:52.700,2021-09-10 00:42:54.700,7.420815,-0.091038
1,2021090900,583,7,2021-09-10T00:42:52.800,41263,105.98,19.28,DE,2021-09-10T00:42:52.700,25511,...,35481,104.10,19.95,TE,11,1,2021-09-10 00:42:52.700,2021-09-10 00:42:54.700,7.353258,-0.083332
2,2021090900,583,8,2021-09-10T00:42:52.900,41263,105.90,19.30,DE,2021-09-10T00:42:52.700,25511,...,35481,104.15,19.95,TE,11,1,2021-09-10 00:42:52.700,2021-09-10 00:42:54.700,7.291982,-0.126674
3,2021090900,583,9,2021-09-10T00:42:53.000,41263,105.79,19.31,DE,2021-09-10T00:42:52.700,25511,...,35481,104.20,19.97,TE,11,1,2021-09-10 00:42:52.700,2021-09-10 00:42:54.700,7.199611,-0.147124
4,2021090900,583,10,2021-09-10T00:42:53.100,41263,105.67,19.33,DE,2021-09-10T00:42:52.700,25511,...,35481,104.27,19.99,TE,11,1,2021-09-10 00:42:52.700,2021-09-10 00:42:54.700,7.093687,-0.141410


In [405]:
strain_assignment_df = assignment_filter_data.merge(new_adjusted_week_pr_df[["nflId_qb","nflId_pr","strain","frameId","gameId","playId","possession_id","week","officialPosition"]], how = "inner", left_on = ["nflId_pr","frame_id","week","possession_id"], right_on = ["nflId_pr","frameId","week","possession_id"])

In [406]:
strain_assignment_df.dropna(inplace=True)

In [410]:
strain_assigment_df_frame = strain_assignment_df.groupby(["nflId_pr","frame_id","possession_id","week"]).apply(lambda x: {pb_id:prob_assign for pb_id,prob_assign in zip(x.nflId,x.prob_assigned)} ).reset_index()

In [414]:
strain_assigment_df_frame.rename(inplace = True, axis = 1, mapper = {0:"blocker_attention"})

In [416]:
strain_design_data = strain_assignment_df.drop_duplicates(["nflId_qb","nflId_pr","strain","officialPosition","frame_id","playId","week","gameId","possession_id"])[["nflId_qb","nflId_pr","strain","officialPosition","frame_id","playId","week","gameId","possession_id"]].merge(strain_assigment_df_frame)

In [417]:
strain_design_data["qb_map"] = strain_design_data["nflId_qb"].apply(lambda x: qb_map[x])

In [418]:
strain_design_data["rusher_map"] = strain_design_data["nflId_pr"].apply(lambda x: rusher_map[x])